# Phase A (Notebook 1): Animated Character & Visual Pipeline (`PhaseA1_Visual_Pipeline.ipynb`)

**Environment Stack**: `Animated_AD` Visual Stack (`PySceneDetect`, `OWLv2`, `SAM 2.1`, `DINOv2` on Colab GPU with native `bfloat16` SDPA FlashAttention).

### Workflow Overview
1. **Cell 1 — Hardware Check, Google Drive Mount & Cache Configuration**:
   - Mounts Google Drive (`/content/drive/MyDrive/MovieRAG`) for **movie inputs, persistent model backups (`models_cache/`), and output checkpoints (`workdir/{movie_id}/`)**.
   - Uses fast **Local SSD (`/content/local_models_cache/`)** for active `HF_HOME` / `TORCH_HOME` (`safetensors` `mmap`) and syncs clean sequential copies to/from Google Drive so models are downloaded only once.
2. **Cell 2 — Clone / Pull Latest Code from GitHub (`/content/MovieRag_repo`)**:
   - Clones your GitHub repository (or pulls the latest commits and hot-reloads `pipeline.*` modules in memory).
3. **Cell 3 — Install Visual Stack Dependencies**:
   - Uses Colab's pre-installed GPU PyTorch and installs `sam2`, `scenedetect`, `decord`, and `transformers`.
4. **Cell 4 — Persistent Drive Model Download & Local SSD Sync**:
   - Syncs or downloads `SAM 2.1` (`sam2.1_hiera_large.pt`), `OWLv2` (`google/owlv2-large-patch14-ensemble`), and `DINOv2` (`dinov2_vitg14_reg` / `dinov2_vitl14_reg`) between Google Drive and Local SSD.
5. **Cell 5 — Stage A1 (`stage1_scenes.py`)**:
   - Shot boundary detection, subdivision of shots $>20\text{s}$ into $15\text{s}$ sub-shots, and grouping into $15\text{--}30\text{s}$ Scene Windows (`shots_and_scenes.json`).
6. **Cell 6 — Stage A2 (`stage2_charbank.py`)**:
   - 75-epoch `DINOv2` contrastive fine-tuning on `char_bank/{movie_id}/` and exemplar `.npz` feature extraction.
7. **Cell 7 — Stage A3 (`stage3_visual_recog.py`)**:
   - Per-shot `TGRP` tracking (resumable via `tgrp_tracks.jsonl`), `DINOv2` track classification (`visual_tracks_per_frame.json`), and **Adaptive Per-Shot Sharp Keyframe Extraction** with pre-drawn character bounding boxes (`keyframes/*.jpg`).
8. **Cell 8 — Handoff Verification & Inline Visualization**.

In [ ]:
# ==============================================================================
# CELL 1: Hardware Check, Google Drive Mount & Cache Setup
# ==============================================================================
import os
import subprocess

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running inside Google Colab; using local paths.")

# 2. Configure Movie & Google Drive Paths
MOVIE_ID = "sintel"                       # @param {type:"string"}
MOVIE_TITLE = "Sintel"                    # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/MovieRAG"  # @param {type:"string"}

VIDEO_PATH = f"{DRIVE_ROOT}/videos/{MOVIE_ID}.mp4"
CHAR_BANK_DIR = f"{DRIVE_ROOT}/char_bank/{MOVIE_ID}"
WORKDIR = f"{DRIVE_ROOT}/workdir/{MOVIE_ID}"
MODELS_CACHE_DIR = f"{DRIVE_ROOT}/models_cache"

# 3. Configure Fast Local SSD Working Cache (synced to/from Google Drive in Cell 4)
#    Keeping active HF_HOME and TORCH_HOME on Local SSD avoids Google Drive FUSE
#    mmap / Xet corruption while still persisting all weights on Drive.
LOCAL_CACHE_ROOT = "/content/local_models_cache"
LOCAL_HF_CACHE = f"{LOCAL_CACHE_ROOT}/huggingface"
LOCAL_OWL_DIR = f"{LOCAL_CACHE_ROOT}/owlv2-large-patch14-ensemble"
LOCAL_TORCH_HUB = f"{LOCAL_CACHE_ROOT}/torch_hub"
LOCAL_SAM2_CKPT = f"{LOCAL_CACHE_ROOT}/sam2.1_hiera_large.pt"

for d in (WORKDIR, MODELS_CACHE_DIR, LOCAL_HF_CACHE, LOCAL_TORCH_HUB):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = LOCAL_HF_CACHE
os.environ["HF_HUB_CACHE"] = f"{LOCAL_HF_CACHE}/hub"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TORCH_HOME"] = LOCAL_TORCH_HUB
os.environ["OWLV2_MODEL_PATH"] = LOCAL_OWL_DIR
SAM2_CHECKPOINT = LOCAL_SAM2_CKPT

# 4. Detect GPU Tier (L4 24GB -> 'giant', T4 16GB -> 'large')
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_info.returncode == 0 and gpu_info.stdout.strip():
    gpu_line = gpu_info.stdout.strip().splitlines()[0]
    print(f"Detected GPU: {gpu_line}")
    if "L4" in gpu_line or "A100" in gpu_line or "24" in gpu_line or "40" in gpu_line or "80" in gpu_line:
        MODEL_SIZE = "giant"
    else:
        MODEL_SIZE = "large"
else:
    print("WARNING: No NVIDIA GPU detected! Make sure Colab runtime is set to GPU (L4 or T4).")
    MODEL_SIZE = "large"

print(f"Selected DINOv2 MODEL_SIZE = '{MODEL_SIZE}'")
print(f"Video Path     : {VIDEO_PATH}")
print(f"Char Bank Dir  : {CHAR_BANK_DIR}")
print(f"Output Workdir : {WORKDIR}")
print(f"Drive Models   : {MODELS_CACHE_DIR}")

In [ ]:
# ==============================================================================
# CELL 2: Clone / Pull Latest Source Code from GitHub (/content/MovieRag_repo)
# ==============================================================================
import importlib
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/MovieRag.git"  # @param {type:"string"}
GITHUB_BRANCH = "main"                                             # @param {type:"string"}
REPO_SUBDIR = ""  # Set to "MovieRag" if your repo root is ctu-flix monorepo, or "" if MovieRag is the repo root
CLONE_DIR = "/content/MovieRag_repo"

# Optional: Load GITHUB_TOKEN from Colab Secrets if the repository is private
github_token = None
if IN_COLAB:
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

auth_repo_url = GITHUB_REPO_URL
if github_token and GITHUB_REPO_URL.startswith("https://github.com/"):
    auth_repo_url = GITHUB_REPO_URL.replace("https://github.com/", f"https://{github_token}@github.com/")

if not os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"Cloning {GITHUB_REPO_URL} (branch: {GITHUB_BRANCH}) into {CLONE_DIR}...")
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, auth_repo_url, CLONE_DIR], check=True)
else:
    print(f"Repository already exists at {CLONE_DIR}; pulling latest changes from origin/{GITHUB_BRANCH}...")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", auth_repo_url], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)

# Auto-detect whether MovieRag is at CLONE_DIR or CLONE_DIR/MovieRag
if REPO_SUBDIR and os.path.isdir(os.path.join(CLONE_DIR, REPO_SUBDIR)):
    PROJECT_ROOT = os.path.join(CLONE_DIR, REPO_SUBDIR)
elif os.path.isdir(os.path.join(CLONE_DIR, "MovieRag", "pipeline")):
    PROJECT_ROOT = os.path.join(CLONE_DIR, "MovieRag")
else:
    PROJECT_ROOT = CLONE_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["PYTHONPATH"] = f"{PROJECT_ROOT}:{PROJECT_ROOT}/Animated_AD:" + os.environ.get("PYTHONPATH", "")

# Reload any already-imported pipeline modules so git pull takes effect immediately
for mod_name in [k for k in list(sys.modules) if k == "pipeline" or k.startswith("pipeline.")]:
    importlib.reload(sys.modules[mod_name])

commit_log = subprocess.run(["git", "-C", CLONE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(f"Active PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Latest Commit      : {commit_log}")

In [ ]:
# ==============================================================================
# CELL 3: Install Animated_AD Visual Environment Stack (Colab Python 3.12+ Ready)
# ==============================================================================
# Uses Colab's pre-installed GPU PyTorch (with native SDPA FlashAttention) and
# installs SAM 2.1, PySceneDetect, Decord, and OWLv2 (transformers).

!pip install -q omegaconf torchmetrics fvcore iopath decord scenedetect opencv-python-headless transformers accelerate huggingface_hub networkx scipy scikit-learn matplotlib pillow tqdm

import os
import site
import sys
import torch
if not os.path.isdir("/content/sam2"):
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2
    !pip install -q -e /content/sam2 --no-build-isolation

site.main()
if "/content/sam2" not in sys.path:
    sys.path.insert(0, "/content/sam2")

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print("Visual environment packages installed successfully.")

In [ ]:
# ==============================================================================
# CELL 4: Persistent Drive Model Download & Local SSD Cache Sync
# ==============================================================================
# Downloads SAM 2.1, OWLv2, and DINOv2 on the first run and backs them up to
# /content/drive/MyDrive/MovieRAG/models_cache/. On subsequent Colab sessions,
# copies the cached weights from Google Drive -> Local SSD (/content/local_models_cache).

import gc
import os
import shutil
import ssl
import urllib.request
import torch
from huggingface_hub import hf_hub_download
from transformers import Owlv2ForObjectDetection, Owlv2Processor

# 1. Bypass expired SSL cert on Meta's dl.fbaipublicfiles.com CDN
_unverified_ctx = ssl._create_unverified_context()
ssl._create_default_https_context = ssl._create_unverified_context
urllib.request.install_opener(
    urllib.request.build_opener(urllib.request.HTTPSHandler(context=_unverified_ctx))
)

DRIVE_OWL_DIR = f"{MODELS_CACHE_DIR}/owlv2-large-patch14-ensemble"
DRIVE_TORCH_CKPTS = f"{MODELS_CACHE_DIR}/torch_hub_checkpoints"
DRIVE_SAM2_CKPT = f"{MODELS_CACHE_DIR}/sam2/sam2.1_hiera_large.pt"

for d in (LOCAL_HF_CACHE, LOCAL_TORCH_HUB, DRIVE_TORCH_CKPTS, os.path.dirname(DRIVE_SAM2_CKPT)):
    os.makedirs(d, exist_ok=True)

torch.hub.set_dir(LOCAL_TORCH_HUB)

# --- [1/3] SAM 2.1 Large Checkpoint (~856 MB) ---
if os.path.isfile(DRIVE_SAM2_CKPT) and os.path.getsize(DRIVE_SAM2_CKPT) > 800 * 1024 * 1024:
    if not os.path.isfile(LOCAL_SAM2_CKPT):
        print("[1/3] Copying cached SAM 2.1 from Google Drive -> Local SSD...")
        shutil.copy2(DRIVE_SAM2_CKPT, LOCAL_SAM2_CKPT)
    else:
        print("[1/3] SAM 2.1 already on Local SSD & Google Drive.")
else:
    print("[1/3] Downloading SAM 2.1 to Local SSD then saving to Google Drive...")
    cached_hf = hf_hub_download(
        repo_id="facebook/sam2.1-hiera-large",
        filename="sam2.1_hiera_large.pt",
        cache_dir=LOCAL_HF_CACHE,
    )
    shutil.copy2(cached_hf, LOCAL_SAM2_CKPT)
    shutil.copy2(LOCAL_SAM2_CKPT, DRIVE_SAM2_CKPT)
SAM2_CHECKPOINT = LOCAL_SAM2_CKPT
print(f"      -> SAM 2.1 ready ({os.path.getsize(SAM2_CHECKPOINT) / (1024**2):.1f} MB)")

# --- [2/3] OWLv2 (google/owlv2-large-patch14-ensemble) ---
drive_owl_safetensors = os.path.join(DRIVE_OWL_DIR, "model.safetensors")
if os.path.isfile(drive_owl_safetensors) and os.path.getsize(drive_owl_safetensors) > 1500 * 1024 * 1024:
    if not os.path.isfile(os.path.join(LOCAL_OWL_DIR, "model.safetensors")):
        print("[2/3] Copying cached OWLv2 from Google Drive -> Local SSD...")
        shutil.copytree(DRIVE_OWL_DIR, LOCAL_OWL_DIR, dirs_exist_ok=True)
    _owl_proc = Owlv2Processor.from_pretrained(LOCAL_OWL_DIR)
    _owl_model = Owlv2ForObjectDetection.from_pretrained(LOCAL_OWL_DIR)
    del _owl_proc, _owl_model
    print("      -> OWLv2 loaded from Drive cache and verified on Local SSD.")
else:
    print("[2/3] Downloading OWLv2 to Local SSD and saving clean copy to Google Drive...")
    _owl_proc = Owlv2Processor.from_pretrained("google/owlv2-large-patch14-ensemble")
    _owl_model = Owlv2ForObjectDetection.from_pretrained("google/owlv2-large-patch14-ensemble")
    os.makedirs(LOCAL_OWL_DIR, exist_ok=True)
    _owl_proc.save_pretrained(LOCAL_OWL_DIR)
    _owl_model.save_pretrained(LOCAL_OWL_DIR)
    del _owl_proc, _owl_model
    if os.path.isdir(DRIVE_OWL_DIR):
        shutil.rmtree(DRIVE_OWL_DIR, ignore_errors=True)
    shutil.copytree(LOCAL_OWL_DIR, DRIVE_OWL_DIR)
    print("      -> OWLv2 saved to Google Drive & Local SSD.")

# --- [3/3] DINOv2 (facebookresearch/dinov2) ---
dinov2_arch = {"small": "vits14", "base": "vitb14", "large": "vitl14", "giant": "vitg14"}[MODEL_SIZE]
dinov2_model_name = f"dinov2_{dinov2_arch}_reg"
dinov2_ckpt_fname = f"dinov2_{dinov2_arch}_reg4_pretrain.pth"
local_dino_ckpt = os.path.join(LOCAL_TORCH_HUB, "checkpoints", dinov2_ckpt_fname)
drive_dino_ckpt = os.path.join(DRIVE_TORCH_CKPTS, dinov2_ckpt_fname)
os.makedirs(os.path.dirname(local_dino_ckpt), exist_ok=True)

if os.path.isfile(drive_dino_ckpt) and os.path.getsize(drive_dino_ckpt) > 50 * 1024 * 1024:
    if not os.path.isfile(local_dino_ckpt):
        print(f"[3/3] Copying cached {dinov2_ckpt_fname} from Google Drive -> Local SSD...")
        shutil.copy2(drive_dino_ckpt, local_dino_ckpt)

print(f"[3/3] Verifying DINOv2 ({dinov2_model_name}) on Local SSD...")
_dino_model = torch.hub.load("facebookresearch/dinov2", dinov2_model_name, pretrained=True, verbose=False)
del _dino_model
if os.path.isfile(local_dino_ckpt) and not os.path.isfile(drive_dino_ckpt):
    print(f"      -> Saving {dinov2_ckpt_fname} to Google Drive for future sessions...")
    shutil.copy2(local_dino_ckpt, drive_dino_ckpt)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"      -> {dinov2_model_name} ready on Local SSD & Google Drive.")
print("\nAll 3 foundation models are verified on Local SSD and backed up to Google Drive!")

In [ ]:
# ==============================================================================
# CELL 5: Stage A1 — Shot Boundary Detection & 15-30s Scene Windowing
# ==============================================================================
import json
from pipeline.stage1_scenes import run_stage1_scenes

stage1_result = run_stage1_scenes(
    video_path=VIDEO_PATH,
    movie_id=MOVIE_ID,
    output_dir=WORKDIR,
    movie_title=MOVIE_TITLE,
    min_scene_sec=12.0,
    max_scene_sec=30.0,
    max_shot_sec=20.0,
    sub_shot_sec=15.0,
)

print("\n--- Stage A1 Summary ---")
print(f"Raw Detected Shots : {stage1_result['num_raw_shots']}")
print(f"Subdivided (>20s)  : {stage1_result['num_subdivided_shots']}")
print(f"Canonical Shots    : {stage1_result['num_shots']}")
print(f"Grouped Scenes     : {stage1_result['num_scenes']}")
if stage1_result["scenes"]:
    print("Sample Scene 0     :", json.dumps(stage1_result["scenes"][0], indent=2))

In [ ]:
# ==============================================================================
# CELL 6: Stage A2 — Visual Character Bank & 75-Epoch DINOv2 Fine-Tuning
# ==============================================================================
from pipeline.stage2_charbank import run_stage2_charbank

OVERWRITE_DINOV2_FINETUNE = False  # @param {type:"boolean"}

stage2_result = run_stage2_charbank(
    movie_id=MOVIE_ID,
    char_bank_dir=CHAR_BANK_DIR,
    output_dir=WORKDIR,
    model_size=MODEL_SIZE,
    epochs=75,
    learning_rate=6e-4,
    temperature=0.07,
    num_per_character=8,
    use_box=True,
    use_mask=False,
    sam2_checkpoint=SAM2_CHECKPOINT,
    overwrite=OVERWRITE_DINOV2_FINETUNE,
)

CKPT_WEIGHTS_PATH = stage2_result["ckpt_weights_path"]
CHAR_FEATURES_DIR = stage2_result["char_features_dir"]
print("\n--- Stage A2 Summary ---")
print(f"Fine-Tuned Weights : {CKPT_WEIGHTS_PATH}")
print(f"Character Bank Dir : {CHAR_FEATURES_DIR}")
print(f"Characters Indexed : {stage2_result['characters']}")

In [ ]:
# ==============================================================================
# CELL 7: Stage A3 — Per-Shot TGRP Tracking, DINOv2 ID & Adaptive Annotated Keyframes
# ==============================================================================
import importlib
import os
import pipeline
import pipeline.stage3_visual_recog

importlib.reload(pipeline)
importlib.reload(pipeline.stage3_visual_recog)
from pipeline.stage3_visual_recog import run_stage3_visual_recog

SHOTS_FILE = os.path.join(WORKDIR, "shots_and_scenes.json")
RESTART_TGRP_FROM_SCRATCH = False  # @param {type:"boolean"}

stage3_result = run_stage3_visual_recog(
    video_path=VIDEO_PATH,
    shots_file=SHOTS_FILE,
    ckpt_weights=CKPT_WEIGHTS_PATH,
    char_features_dir=CHAR_FEATURES_DIR,
    sam2_checkpoint=SAM2_CHECKPOINT,
    output_dir=WORKDIR,
    model_size=MODEL_SIZE,
    mask_classify=False,
    visual_prompt_thresh=0.45,
    voice_harvest_thresh=0.60,
    max_boxes_per_seed=8,
    restart=RESTART_TGRP_FROM_SCRATCH,
)

print("\n--- Stage A3 Summary ---")
print(f"Total Shots Tracked        : {stage3_result['num_shots']}")
print(f"Total Scenes Updated       : {stage3_result['num_scenes']}")
print(f"Total Annotated Keyframes  : {stage3_result['total_keyframes']}")
print(f"Track Classification Stats : {stage3_result['track_stats']}")

In [ ]:
# ==============================================================================
# CELL 8: Handoff Artifact Verification & Sample Annotated Keyframe Inspection
# ==============================================================================
import json
import os
import matplotlib.pyplot as plt
from PIL import Image

required_files = [
    os.path.join(WORKDIR, "shots_and_scenes.json"),
    os.path.join(WORKDIR, "visual_tracks_per_frame.json"),
    os.path.join(WORKDIR, "classified_tracks.json"),
    os.path.join(WORKDIR, "character_glossary.json"),
]

print("=== Notebook 1 -> Notebook 2 Handoff Verification ===")
for path in required_files:
    exists = os.path.isfile(path)
    size_kb = os.path.getsize(path) / 1024.0 if exists else 0.0
    print(f"[{'OK' if exists else 'MISSING'}] {path} ({size_kb:.1f} KB)")

with open(os.path.join(WORKDIR, "shots_and_scenes.json"), "r", encoding="utf-8") as f:
    scenes_data = json.load(f)

# Collect up to 4 annotated keyframes that have visible character bounding boxes
annotated_samples = []
for scene in scenes_data.get("scenes", []):
    for kf in scene.get("keyframes", []):
        if kf.get("characters_visible"):
            annotated_samples.append((scene["scene_id"], kf))
            if len(annotated_samples) >= 4:
                break
    if len(annotated_samples) >= 4:
        break

if annotated_samples:
    fig, axes = plt.subplots(1, len(annotated_samples), figsize=(5 * len(annotated_samples), 4))
    if len(annotated_samples) == 1:
        axes = [axes]
    for ax, (scene_id, kf) in zip(axes, annotated_samples):
        img_path = os.path.join(WORKDIR, kf["path"])
        if os.path.isfile(img_path):
            ax.imshow(Image.open(img_path))
        legend_str = ", ".join(
            f"{c['name']} ({c['color']}: {c['score']:.2f})" for c in kf["characters_visible"]
        )
        ax.set_title(f"{scene_id} | t={kf['timestamp']:.2f}s\n{legend_str}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No keyframes with character detections above threshold found to display.")